## 1. Load the loans and look for gaps
Load *library_loans.csv* with `parse_dates` for both date columns. Print the shape and the dtypes. Then count the missing values in every column, and show them as percent. Which 2 columns have gaps, and how many?

In [1]:
import pandas as pd

loans_raw = pd.read_csv("data/library_loans_clean.csv", parse_dates=["loan_date", "return_date"])
loans = loans_raw.copy()
members_raw = pd.read_csv("data/library_members.csv")
members = members_raw.copy()

# loans (600, 8)
print(loans_raw.shape)

# members (148, 3)
print(members_raw.shape)

print(loans_raw.dtypes)

print(members_raw.dtypes)

# 3 N/A in loan_date (0.5%), 36 N/A in return_date (6%)
print(round(loans_raw.isna().mean() * 100, 1))

# 12 N/A in branch (8.1%), 18 N/A in birth_year (12.2%)
print(round(members_raw.isna().mean() * 100, 1))

(600, 8)
(148, 3)
loan_id                   str
member_id                 str
title                     str
genre                     str
loan_date      datetime64[us]
return_date    datetime64[us]
late_fee              float64
returned                 bool
dtype: object
member_id         str
branch            str
birth_year    float64
dtype: object
loan_id        0.0
member_id      0.0
title          0.0
genre          0.0
loan_date      0.5
return_date    6.0
late_fee       0.0
returned       0.0
dtype: float64
member_id      0.0
branch         8.1
birth_year    12.2
dtype: float64


## 2. Two kinds of missing
Show the loans where *loan_date* is missing. Then look at *return_date*: how many are missing and the book is back (returned is True), and how many are missing and the book is not back? Which of the 2 kinds is "missing for a reason", and which is "missing by accident"? Create a column *has_return_date* with `notna()` and count it.

In [2]:
# 3 loan_date are missing
no_loan_date = loans["loan_date"].isna()
print(loans[no_loan_date])

# 6 return_date are missing while returned = true
# there has to be a return date if a book has been returned, therefore this is wrong
missing_and_returned = loans["return_date"].isna() & (loans["returned"] == True)
print(missing_and_returned.sum())


# 30 return_date are missing while returned = false
# there cant be a return date if a book hasnt been returned, this is right
missing_and_not_returned = loans["return_date"].isna() & (loans["returned"] == False)
print(missing_and_not_returned.sum())

# 564 loans have a return date, while 36 haven't
loans["has_return_date"] = loans["return_date"].notna()
print(loans["has_return_date"].value_counts())

     loan_id member_id               title     genre loan_date return_date  \
462  L-30463     M0007  Blue Moon Protocol    Sci-fi       NaT  2026-05-24   
467  L-30468     M0018  Blue Moon Protocol    Sci-fi       NaT  2026-06-08   
571  L-30572     M0090  Mimmi and the Moon  Children       NaT  2026-06-26   

     late_fee  returned  
462       0.0      True  
467       0.0      True  
571       0.0      True  
6
30
has_return_date
True     564
False     36
Name: count, dtype: int64


## 3. Look at the members
Load *library_members.csv*, make a copy and print the shape. Run `info()` and `describe()` on *birth_year*. Something is wrong with the minimum and the maximum. Count how many members have each strange value, and check for duplicates in the whole row and in *member_id*. Write the list of problems as a comment.

In [3]:
# 8 members have birth_year 1900, might be a placeholder
# 3 members have birth_year 2030, which is impossible
# 4 members are exact duplicates, including the member_id
# 18 members have no value in birth_year
# 12 members have no value in branch

# 148 rows & 3 columns
print(members.shape)

members["birth_year"].info()

print(members["birth_year"].describe())

# there are 8 people born in 1900
print((members["birth_year"] == 1900).sum())

# there are 3 people born in 2030
print((members["birth_year"] == 2030).sum())

# there are 4 duplicate members
print(members.duplicated().sum())

# there are 4 members with duplicate id
print(members["member_id"].duplicated().sum())

(148, 3)
<class 'pandas.Series'>
RangeIndex: 148 entries, 0 to 147
Series name: birth_year
Non-Null Count  Dtype  
--------------  -----  
130 non-null    float64
dtypes: float64(1)
memory usage: 1.3 KB
count     130.000000
mean     1983.384615
std        25.158223
min      1900.000000
25%      1978.250000
50%      1988.000000
75%      1996.750000
max      2030.000000
Name: birth_year, dtype: float64
8
3
4
4


## 4. Placeholders and impossible values
Nobody in this library is born in 1990, and nobody is born in the future. Use `where` and `between(1920, 2015)` to turn those values into missing values. How many birth years are missing now? Run `describe()` again and compare it with before.

In [4]:
members["birth_year"] = members["birth_year"].where(members["birth_year"].between(1920, 2015))

# 29 birth_year are now NaN
print(members["birth_year"].isna().sum())

# lowest birth_year is now 1957 and highest is 2010
# birt_year count went down from 130 to 119
# mean went from 1983.4 to 1987.8

print(members["birth_year"].describe())

29
count     119.000000
mean     1987.815126
std        12.062666
min      1957.000000
25%      1979.000000
50%      1988.000000
75%      1996.500000
max      2010.000000
Name: birth_year, dtype: float64


# 5. Remove, fill, or leave
(a) Without saving the result, see how many rows are left with dropna(), with `dropna(subset=["branch"])` and with `dropna(subnet=["birth_year"])`.

(b) Fill the missing *branch* with `"Unknown"` and count the branches.

(c) Try filling the missing *birth_year* with the median, on the side, in a separate variable. Compare the mean and the standard deviation before and after. What happens to the spread, and why? What do you decide to do with *birth_year*?

In [5]:
# 107 rows are left after dropping rows where there's no value
print(members.dropna().shape)

# 136 rows are left after dropping rows in branch where there's no value
print(members.dropna(subset="branch").shape)

# 119 rows are left after dropping rows in birth_year where there's no value
print(members.dropna(subset="birth_year").shape)

# fills rows with "Unknown" where value is empty
members["branch"] = members["branch"].fillna("Unknown")

print(members["branch"].value_counts())

# fills rows with median where value is empty
filled_birth_year = members["birth_year"].fillna(members["birth_year"].median())

# mean is 1987.81
print(members["birth_year"].mean())

# median is 1988
print(members["birth_year"].median())

# std is 12
print(members["birth_year"].std())

# mean is 1987.85
print(filled_birth_year.mean())

# median is 1988
print(filled_birth_year.median())

# std is 10.8
print(filled_birth_year.std())

# std drops because all 29 filled members get the same year, 1988
# i would leave birth_year as NaN, since filling it would invent fake data

(107, 3)
(136, 3)
(119, 3)
branch
Central    40
South      34
North      32
East       17
Harbour    13
Unknown    12
Name: count, dtype: int64
1987.81512605042
1988.0
12.062665744223288
1987.8513513513512
1988.0
10.807760417230606


## 6. Merge, and check
Print the shape of both tables. Merge the loans with the members with `how="left"` and without `validate`. How many rows do you get, and why is it not 600? Calculate the total of *late_fee* before and after the merge. Then merge again with `validate="many_to_one"` and read the error from the bottom up. The merge with `validate` should give a MergeError.

In [6]:
# 600 rows, 9 columns
print(loans.shape)

# 148 rows, 3 columns
print(members.shape)

# sum is 7487.5
print(loans["late_fee"].sum())

# 616 rows, 11 columns
# 4 members are in members twice, each of their loans is matched twice
merged_tables = loans.merge(members,
                            on="member_id",
                            how="left")
print(merged_tables.shape)

# sum is 7725, which is 237.5 more than before
print(merged_tables["late_fee"].sum())

# gives error because there are members with the same member_id
print(loans.merge(members,
                  on="member_id",
                  how="left",
                  validate="many_to_one",
                  indicator=True).shape)

(600, 9)
(148, 3)
7487.5
(616, 11)
7725.0


MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 member_id
    M0015
    M0033
    M0065
    M0071 ...

## 7. Fix the key, and find who is missing
Remove the duplicates from *members* and merge again with `validate="many_to_one"` and `indicator=True`. Check the shape and the total of *late_fee*. How many loans have *left_only*, how many different members is that, and how many kr in late fees? What would an inner merge have given for the shape and the total?

In [7]:
members = members.drop_duplicates().reset_index(drop=True)

loans_merged = loans.merge(members,
                           on="member_id",
                           how="left",
                           validate="many_to_one",
                           indicator=True)

# 600 rows and 12 columns
print(loans_merged.shape)

# sum is 7487.5 kr
print(loans_merged["late_fee"].sum())

# 31 loans have left_only, 0 loans have right_only and 569 loans have both
print(loans_merged["_merge"].value_counts())

# of the 31 left_only loans, 6 members don't exist in the members table
left_only_loans = loans_merged["_merge"] == "left_only"
print(loans_merged[left_only_loans]["member_id"].nunique())

# 347.5 kr
print(loans_merged[left_only_loans]["late_fee"].sum())

inner_loans_merged = loans.merge(members,
                                 on="member_id",
                                 how="inner",
                                 validate="many_to_one")

# 569 rows and 11 columns
print(inner_loans_merged.shape)

# sum is 7140 kr
print(inner_loans_merged["late_fee"].sum())

(600, 12)
7487.5
_merge
both          569
left_only      31
right_only      0
Name: count, dtype: int64
6
347.5
(569, 11)
7140.0
